# Модель A: классификатор дизайна исследования (обучение в YC, plan §7)

LR-голова на замороженных эмбеддингах e5-small. Заполняет `design` там, где Publication Type молчит (`UNKNOWN`). Инференс в пайплайне — `src/adapters/classify/design_clf.py` (`ClassifierPort`).

**Данные — реальные, из PubMed:** метки = кураторские `PublicationType` от NLM (RCT / review / meta-analysis / case report / observational). Сборка: `training/build_design_dataset.py` → `design_labels.csv`. Это тот же принцип, что у Cochrane RCT Classifier: модель учится читать текст абстракта и восстанавливать дизайн там, где тег отсутствует. Метки должны совпадать со значениями `StudyDesign`: `rct`, `observational`, `review`, `case_report`.

Запуск: DataSphere → project → Run All (минуты на CPU). **Чекпойнт:** `design_clf.joblib` → Object Storage → `data/models/design_clf.joblib` (путь, который читает адаптер).

In [ ]:
# В DataSphere раскомментируйте при необходимости:
# %pip install -q sentence-transformers scikit-learn joblib

In [ ]:
# РЕАЛЬНЫЕ данные из PubMed (метки = кураторские PublicationType от NLM).
# Собираются `training/build_design_dataset.py` -> design_labels.csv (text,label).
# Если CSV нет — крошечный демо-сэмпл (только чтобы ячейки прогонялись).
import csv
from pathlib import Path

csv_path = Path("design_labels.csv")
if csv_path.exists():
    with csv_path.open(encoding="utf-8") as f:
        data = [(r["text"], r["label"]) for r in csv.DictReader(f)]
    print(f"реальные данные: {len(data)} строк из {csv_path}")
else:
    data = [
        ("Randomized controlled trial comparing drug A vs placebo, double-blind.", "rct"),
        ("We randomly assigned 200 patients to two arms in a phase III trial.", "rct"),
        ("Retrospective cohort study of registry data over 10 years.", "observational"),
        ("A prospective observational cohort followed for survival outcomes.", "observational"),
        ("This systematic review and meta-analysis pooled 12 studies.", "review"),
        ("Narrative review of immunotherapy in melanoma.", "review"),
        ("Case report of a rare adverse event in one patient.", "case_report"),
        ("We describe a single case of unexpected response.", "case_report"),
    ]
    print(f"демо-сэмпл: {len(data)} строк (положи design_labels.csv для реальных метрик)")
texts = [t for t, _ in data]
labels = [y for _, y in data]
len(texts)

In [ ]:
from sentence_transformers import SentenceTransformer

# Тот же энкодер, что и в адаптере (src/adapters/classify/design_clf.py).
encoder = SentenceTransformer("intfloat/multilingual-e5-small")
X = encoder.encode(texts, normalize_embeddings=True)
X.shape

In [ ]:
# Честная оценка на hold-out (крит. 9), не на train. На реальных данных (~1000 строк)
# stratified hold-out работает штатно. На крошечном демо (2 примера/класс) отдельный
# hold-out со всеми классами невозможен -> fallback на out-of-fold через k-fold.
from collections import Counter

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

min_class = min(Counter(labels).values())
if min_class >= 8:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, labels, test_size=0.25, random_state=0, stratify=labels
    )
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    print(classification_report(y_te, clf.predict(X_te), digits=3))
else:
    cv = StratifiedKFold(n_splits=min(2, min_class), shuffle=True, random_state=0)
    y_oof = cross_val_predict(LogisticRegression(max_iter=1000), X, labels, cv=cv)
    print("демо: out-of-fold (цифры условны, нужен gold)")
    print(classification_report(labels, y_oof, digits=3))

In [ ]:
import joblib

# Финальный чекпойнт — на ВСЕХ данных (hold-out выше был только для честной оценки).
clf = LogisticRegression(max_iter=1000).fit(X, labels)
joblib.dump(clf, "design_clf.joblib")
print("сохранён design_clf.joblib")
# Чекпойнт -> Object Storage -> data/models/design_clf.joblib (путь адаптера).
# import boto3
# s3 = boto3.client('s3', endpoint_url='https://storage.yandexcloud.net')
# s3.upload_file('design_clf.joblib', '<bucket>', 'models/design_clf.joblib')